# Project 3: Prediction

In [ ]:
# FIXME: Enter your group name here
GROUP = "X"

## 3.0 Preparation

Ensure that you run this notebook in your project environment created for the previous project. *On Windows: ensure that you are using WSL!*

Import the necessary Python modules. We will re-use the FeniCS interface from the previous project. For that to work, you need to place this Jupyter notebook in the `project` directory (the directory containing `fem_main.py`).

In [ ]:
!pip install tensorflow

In [ ]:
from pathlib import Path

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import random
import tensorflow as tf

from fem.interface.fenics_interface_solved import FenicsInterface
from fem.base import grids

## 3.1 Create a data-generating function

The dataset should consist of multiple samples of features, some of which are used as input and output for our machine-learning problem.
Based on the subsequent tasks in this notebooks, the input features must be varied.

We will use the elastoplastic finite-element cube from Project 1 and 2 as data-generating function. As in Project 2, apply just a displacement-controlled loading and no unloading and remember to assemble the histories into single numpy arrays. Compute the same derived values as in Project 2. The data-generating functions should return a dictionary of numpy arrays for the each sample of varied input parameters.

In [ ]:
# Do not change this cell!
#   This function is a useful helper to apply grid encoding to the frame data. The lines
#   using it later are already prepared for you. You do not need to use it anywhere else
#   in the notebook or modify this function.
def apply_grid_encoding(frame, grid_shape):
    """Grid encode the frame data."""
    Xi = frame["Xi"]
    origin = np.min(Xi, axis=0)
    geometry = tuple(np.max(Xi, axis=0) - np.min(Xi, axis=0))
    grid_coords = grids.generate_grid(geometry, grid_shape) + origin
    reduced_frame = {
        k: v
        for k, v in frame.items()
        if "fenics" not in k and "bc_" not in k and isinstance(v, np.ndarray)
    }
    del reduced_frame["Pi"]
    _, _, grid_encoded = grids.grid_encode(grid_coords, grid_shape, Xi, reduced_frame)
    return grid_encoded

Use the provided function and signature for `solve_fem(...)` and complete the physics-based finite-element simulation code within.

The result dictionary is currently missing Young's (elastic) modulus `E` and the tensile yield limit `sig0`.
In addition, extract the deformation imposed on the elastoplastic cube at the boundaries and store it in `U_boundary`. The tensor `U_boundary` should have only values other than zero in the 0 and -1 indices of the spatial axes (27, 27).

Add all three values to the result dictionaries as they will be needed to prepare the data for the prediction. 

In [ ]:
def solve_fem(E=7e10, u0=1e-3, nu=0.3, rho=0.0, sig0=2.50e8, Et=1e-9):
    """Solve the elastoplastic cube problem using the FenicsInterface.

    Args:
        E (float, optional): Young's (elastic) modulus. Defaults to 7e10.
        u0 (float, optional): Maximum displacement at the left and right edges of the cube. Defaults to 1e-3.
        nu (float, optional): Poisson's ratio. Defaults to 0.3.
        rho (float, optional): Density of the material. Defaults to 0.0.
        sig0 (float, optional): Tensile yield limit. Defaults to 2.50e8.
        Et (float, optional): Tangent modulus for kinematic hardening. Defaults to 1e-9.

    Returns:
        dict: A dictionary containing the results of the simulation as feature tensors.
    """

    # FIXME Run the simulations using the FenicsInterface



    U = None
    F = None
    S = None
    Mises = None

    # FIXME Compute U boundary from a copy of U
    U_boundary = None

    results = {
        # FIXME Add E, sig0 and U_boundary
        "U": U,  # Displacement tensor of shape (T, X, Y, 2)
        "F": F,  # Force tensor of shape (T, X, Y, 2)
        "S": S,  # Stress tensor of shape (T, X, Y, 6)
        "Mises": Mises,  # Von Mises stress tensor of shape (T, X, Y, 1)
    }

    # FIXME Compute non-local and non-directional features from the results.
    #  Compute the magnitudes of the displacement and force vectors at each grid point
    #  for each time frame and store them in the results dictionary.



    results["max_u"] = None  # Max displacement magnitude tensor of shape (T, 1)
    results["min_u"] = None  # Min displacement magnitude tensor of shape (T, 1)
    results["max_f"] = None  # Max force magnitude tensor of shape (T, 1)
    results["min_f"] = None  # Min force magnitude tensor of shape (T, 1)
    results["max_mises"] = None  # Max Von Mises stress tensor of shape (T, 1)
    results["min_mises"] = None  # Min Von Mises stress tensor of shape (T, 1)

    return results

## 3.2 Generate data

Run the data-generating function a few times. Vary the input features `E`, `sig0` and `u0` in the intervals given in the cell directly below. Select a suitable randomization method (Use the `numpy` or `random` modules). 

Collect the results of each simulation (samples) in a suitable data structure. Finally, store that data structure using `np.save`.

After you are happy with the generated data, you should increase the number of samples to `num_samples = 100` or higher. Use the `RECREATE` flag to only execute the time-intensive data generation when needed. 

In [ ]:
RECREATE = True  # Set to True to re-run the simulation

results_file = Path(f"RESULTS/samples_group{GROUP}.npy")
results_file.parent.mkdir(parents=True, exist_ok=True)

# Define dataset and variable ranges
num_samples = 5
E_min, E_max = 6e10, 8e10 
u0_min, u0_max = 1e-3, 4e-3 
sig0_min, sig0_max = 2.50e8, 2.75e8

In [ ]:
# Check if the results file exists and if we need to recreate it
if RECREATE or not results_file.exists():
    # Run multiple iterations to generate a first dataset
    samples_list = []
    for _ in tqdm(range(num_samples), desc="Data Generation"):
        # FIXME Use a randomization function within the boundaries
        # Call the solve_fem function with the random values
        # Store the result in a list
        E_rand = None
        u0_rand = None
        sig0_rand = None

        results = {}
        
        samples_list.append(results)
        
    # FIXME Save the samples in a npy file
    print(results_file)
else:
    # Load results results
    samples_list = np.load(results_file, allow_pickle=True)

In [ ]:
# Check the loaded/generated data
for key, value in samples_list[0].items():
    print(f"{key}: {value.shape if isinstance(value, np.ndarray) else value}")

assert "E" in samples_list[0]
assert "sig0" in samples_list[0]
assert "U_boundary" in samples_list[0].keys()
assert not np.any(samples_list[0]["U_boundary"][:, 1:-1, 1:-1, :])

# 3.3 Prediction of dense neural networks

Predict the outputs `max_f` and `max_mises` from the inputs `max_u`, `E`, and `sig0`.


**(a) Assemble the input and output features and tensors.**

For the prediction, the inputs and target features need to be assembled as tensors (`np.ndarrays`) of multiple samples. The first axis of these tensors corresponds to the sample axis.
Example for `max_u_tensor` (N=100, T=6, F=1). You may need to expand scalar features to the sequence shape.



In [ ]:
# Create empty list to store the data
#   Note: there are faster ways to do this, but we keep the code simple for clarity
max_u_list = []
E_list = []
sig0_list = []
max_f_list = []
max_mises_list = []

# Iterate over the samples and extract the relevant data
for sample in samples_list:
    
    # FIXME Extract the relevant data from the sample
    max_u = None
    E = None
    sig0 = None
    max_f = None
    max_mises = None

    # FIXME Expand the scalars to match the structural shapes of the other features
    E = None
    sig0 = None
    
    # Store the data
    max_u_list.append(max_u)
    E_list.append(E)
    sig0_list.append(sig0)
    max_f_list.append(max_f)
    max_mises_list.append(max_mises)

Assemble the input and output tensors `X` and `Y` by concatenating the feature tensors along the last feature axis F (`np.concatenate`). 

In [ ]:
# FIXME Convert features to numpy arrays of compatible shapes
max_u_tensor = None  # shape: (N, 6, 1)
E_tensor = None  # shape: (N, 6, 1)
sig0_tensor = None  # shape: (N, 6, 1)
max_f_tensor = None  # shape: (N, 6, 1)
max_mises_tensor = None  # shape: (N, 6, 1)

# FIXME Assemble the input features
X = None

# FIXME Assemble the target features
Y = None

**(b) Normalize the input and output data**

 Use the sklearn `StandardScaler` to scale inputs `X` and outputs `Y` to a standard deviation of +-1. 
 Ensure that the relative variance across the temporal axis T=6 (and later spatial axes X=Y=27) remain unchanged.
 Remember to restore the structural axes (T, X, Y) afterwards

 (Tip: use `np.reshape` to collapse all non-feature axes into the samples axis and restore the shape afterwards).

In [ ]:
# FIXME Standardize and reshape the values



X0 = None
Y0 = None

**(c) Split the dataset**

Perform a suitable dataset split for the size of the dataset you generated.
The sklearn function `train_test_split` may help you.

In [ ]:
# FIXME Train-test split (keep temporal structure)
X_train, X_test, Y_train, Y_test = X0, X0, Y0, Y0

**(d) Model development**

Develop a neural network model. Either use `tf.keras.Model` or `tf.keras.Sequential`.

To define the architecture in the model, tensorflow's `layers` module may help you (e.g., `layers.Dense`, `layers.Input`).
Chose reasonable activation functions for your machine-learning problem.
You will need to operate on individual time steps. For dense layers to be applied to individual frames of a time step, wrap them in a `layers.TimeDistributed` layer.

Compile the model and select an apropriate optimzer, loss, and metric function.

In [ ]:
# FIXME: Define a time-distributed dense neural network


model = None


In [ ]:

model.summary()

**(e) Train the Model**

Train the model until you achieve convergence and generalization. Remember to validate your training performance against a validation dataset.

In the process, you may need to change your model hyperparameters (number of layers, number of units per layer, activation functions, learning rate, batch size, ...).

Do not evaluate your test set yet!

In [ ]:
# FIXME: Train the model
history = None

**(e) Plot the training and validation performance**

Plot your loss and metrics using `subplot` from `matplotlib`

In [ ]:
# FIXME Plot training & validation loss
plt.figure(figsize=(12, 5))

# Loss
plt.subplot(1, 2, 1)



plt.title('Loss over epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# metric
if 'mae' in history.history:
    plt.subplot(1, 2, 2)

    
    
    plt.title('MAE over epochs')
    plt.xlabel('Epoch')
    plt.ylabel('MAE')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()

**(f) Postprocess the outputs on the test set**

Retransform the predicted data to it's original scale using `inverse_transform`. Also, make sure to use `reshape` so that the final shape is correct.

In [ ]:
# Evaluate the model
Y_pred0 = None

# FIXME Inverse transformation
Y_pred = None
Y_true = None

**(g) Evaluate the model predictions on the test set**

Evaluate max_f and max_mises individually using `mean_absolute_error`, `mean_squared_error`, `r2_score`.

In [ ]:
# Reshape the true and predicted array
Y_true_flat = None
Y_pred_flat = None

# FIXME Evaluation of max_f
mae_max_f = 0.0
rmse_max_f = 0.0
r2_max_f = 0.0

# FIXME Evaluation of max_mises
mae_max_mises = 0.0
rmse_max_mises = 0.0
r2_max_mises = 0.0

print(f"max_f: \n   MAE: {mae_max_f:.4f}, RMSE: {rmse_max_f:.4f}, R2: {r2_max_f:.4f}")
print(f"max_mises: \n   MAE: {mae_max_mises:.4f}, RMSE: {rmse_max_mises:.4f}, R2: {r2_max_mises:.4f}")

Plot residual plots of actual vs predicted values for each of the output features, `max_f` and `max_u`.

In [ ]:
# Plot max_f
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(Y_true_flat[:, 0], Y_pred_flat[:, 0], alpha=0.5, c="blue")
plt.plot(
    [Y_true_flat[:, 0].min(), Y_true_flat[:, 0].max()],
    [Y_true_flat[:, 0].min(), Y_true_flat[:, 0].max()],
    "r--",
    label="Perfect Prediction",
)
plt.xlabel("True max_f")
plt.ylabel("Predicted max_f")
plt.title("Actual vs Predicted max_f")
plt.legend()
plt.grid(True)

# Plot max_mises
plt.subplot(1, 2, 2)
plt.scatter(Y_true_flat[:, 1], Y_pred_flat[:, 1], alpha=0.5, c="green")
plt.plot(
    [Y_true_flat[:, 1].min(), Y_true_flat[:, 1].max()],
    [Y_true_flat[:, 1].min(), Y_true_flat[:, 1].max()],
    "r--",
    label="Perfect Prediction",
)
plt.xlabel("True max_mises")
plt.ylabel("Predicted max_mises")
plt.title("Actual vs Predicted max_mises")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

Create plots to visualize the first 5 actual vs predicted outputs for both `max_f` and `max_mises`.

In [ ]:
# Number of samples to plot
num_plot = 5
time_steps = Y_true.shape[1]  # 6

for i in range(num_plot):
    plt.figure(figsize=(10, 4))
    
    # FIXME Plot max_f
    plt.subplot(1, 2, 1)


    plt.title(f"Sample {i}: max_f")
    plt.xlabel("Time step")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True)
    
    # FIXME Plot max_mises
    plt.subplot(1, 2, 2)


    plt.title(f"Sample {i}: max_mises")
    plt.xlabel("Time step")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

## 3.4 Boundary displacement to stress field

Predict the von-Mises stress field `Mises` from the displacements on the boundary `U_boundary`, Young's modulus `E`, and the tensile yield limit `sig0`.

**(a) Assemble the input and output features and tensors.**

For the prediction, the inputs and target features need to be assembled as tensors (`np.ndarrays`) of multiple samples. The first axis of these tensors corresponds to the sample axis.
Example for `U_boundary_tensor` (N=100, T=6, X=27, Y=27, F=2). You may need to expand scalar features to the spatiotemporal shape.

In [ ]:
# Create empty list
#   Note: there are faster ways to do this, but we keep the code simple for clarity
E_list = []
sig0_list = []
U_boundary_list = []
Mises_list = []

# Iterate over the samples and extract the relevant data
for sample in samples_list:
    
    # FIXME Extract the relevant data from the sample
    U_boundary = None
    Mises = None
    E = None
    sig0 = None
    
    # FIXME Expand the scalars to match the structural shapes of the other features
    E = None
    sig0 = None
    
    U_boundary_list.append(U_boundary)
    Mises_list.append(Mises)
    E_list.append(E)
    sig0_list.append(sig0)

Assemble the input and output tensors `X` and `Y` by concatenating the feature tensors along the last feature axis F (`np.concatenate`). 

In [ ]:
# FIXME Convert the data to numpy array
U_boundary_tensor = None  # (N, 6, 27, 27, 2)
E_tensor = None  # (N, 6, 27, 27, 1)
sig0_tensor = None  # (N, 6, 27, 27, 1)
Mises_tensor = None  # (N, 6, 27, 27, 6)

# FIXME Assemble input features
X = None

# FIXME Assemble target features:
Y = None

**(b) Normalize the input and output data**

 Use the sklearn `StandardScaler` to scale inputs `X` and outputs `Y` to a standard deviation of +-1. 
 Ensure that the relative variance across the temporal axis T=6 (and later spatial axes X=Y=27) remain unchanged.
 Remember to restore the structural axes (T, X, Y) afterwards

 (Tip: use `np.reshape` to collapse all non-feature axes into the samples axis and restore the shape afterwards).

In [ ]:
# FIXME Standardize and reshape the values






X0 = None
Y0 = None

**(c) Split the dataset**

Perform a suitable dataset split for the size of the dataset you generated.
The sklearn function `train_test_split` may help you.

In [ ]:
# FIXME Train-test split
X_train, X_test, Y_train, Y_test = X0, X0, Y0, Y0

**(d) Model development**

Develop a neural network model. Either use `tf.keras.Model` or `tf.keras.Sequential`.

To define the architecture in the model, tensorflow's `layers` module may help you (e.g., `layers.Conv2D`, `layers.MaxPool2D`, `layers.AveragePoling2D`).
Chose reasonable activation functions for your machine-learning problem.
You will need to operate on individual time steps. For layers to be applied to individual frames of a time step, wrap them in a `layers.TimeDistributed` layer.

Compile the model and select an apropriate optimzer, loss, and metric function.

In [ ]:
# FIXME Develop a TimeDistributed CNN model





model = None

In [ ]:
model.summary()

**(e) Train the Model**

Train the model until you achieve convergence and generalization. Remember to validate your training performance against a validation dataset.

In the process, you may need to change your model hyperparameters (number of layers, number of units per layer, activation functions, learning rate, batch size, ...).

Do not evaluate your test set yet!

In [ ]:
# Train
history = None

**(e) Plot the training and validation performance**

Plot your loss and metrics using `subplot` from `matplotlib`

In [ ]:
# FIXME Plot training & validation loss and MAE
plt.figure(figsize=(12, 5))

# FIXME Plot Loss
plt.subplot(1, 2, 1)



plt.title('MSE Loss over epochs')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)

# FIXME Plot metric
if 'mae' in history.history:
    plt.subplot(1, 2, 2)


    plt.title('MAE over epochs')
    plt.xlabel('Epoch')
    plt.ylabel('MAE')
    plt.legend()
    plt.grid(True)

plt.tight_layout()
plt.show()

**(f) Postprocess the outputs on the test set**

Retransform the predicted data to it's original scale using `inverse_transform`. Also, make sure to use `reshape` so that the final shape is correct.

In [ ]:
# Predict
Y_pred0 = None

# FIXME Inverse scale
num_features = Y_pred0.shape[-1]  # Number of features in the output
Y_pred = None
Y_true = None

**(g) Evaluate the model predictions on the test set**

Evaluate the predicitons against the targets using `mean_absolute_error`, `mean_squared_error`, `r2_score`.

In [ ]:
# FIXME Flatten spatial and time dims to compute metrics over all pixels and frames
Y_true_flat = None
Y_pred_flat = None

# FIXME Compute overall metrics:
mae_overall = 0.0
rmse_overall = 0.0
r2_overall = 0.0
print(f"\nOverall metrics: MAE={mae_overall:.4f}, RMSE={rmse_overall:.4f}, R2={r2_overall:.4f}")

Plot residual plots of actual vs predicted values for each of the output features.

In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(Y_true_flat, Y_pred_flat, alpha=0.3, s=1)
plt.plot([Y_true_flat.min(), Y_true_flat.max()], [Y_true_flat.min(), Y_true_flat.max()], 'r--', label='Ideal fit')
plt.xlabel('Actual values')
plt.ylabel('Predicted values')
plt.title('Actual vs Predicted values')
plt.legend()
plt.grid(True)
plt.show()

Create a plot to visualize the first 5 actual sample vs predicted for the last timeframe.

In [ ]:
num_samples = 5
time_index = -1  # pick time step 0 for visualization

for sample_idx in range(num_samples):
    plt.figure(figsize=(8, 3))
    actual_field = None
    predicted_field = None

    # FIXME Actual
    plt.subplot(1, 2, 1)
    plt.imshow(actual_field, cmap='viridis')
    plt.colorbar()
    plt.title(f'Sample {sample_idx}, Mises - Actual')

    # FIXME Predicted
    plt.subplot(1, 2, 2)
    plt.imshow(predicted_field, cmap='viridis')
    plt.colorbar()
    plt.title(f'Sample {sample_idx}, Mises - Predicted')

    plt.tight_layout()
    plt.show()